# WEEK 3 – CLEANING SPRINT

## Objective
To improve dataset quality by identifying and handling missing values and investigating potential outliers.

## Activities Performed
- Created a separate copy of the original dataset.
- Identified missing values.
- Applied median imputation.
- Applied mode imputation.
- Detected potential outliers using the Z-score method.
- Used box plots for visual outlier identification.
- Verified the cleaned dataset.

## Outcome
The cleaned dataset contained zero missing values and zero duplicate rows.

## 3.0 Importing Libraries and Loading the Original Dataset

Run this setup cell first. Since this notebook is stored inside the Week 3 folder, the relative path points to the original CSV in the repository's `Dataset` folder. If you run the notebook somewhere else, update the path accordingly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

dataset_path = Path('drug_consumption_missing.csv')

if not dataset_path.exists():
    raise FileNotFoundError(
        f'Dataset not found at {dataset_path}. '
        'Check that Original_Dataset.csv is uploaded in the Dataset folder.'
    )

df = pd.read_csv(dataset_path)
print('Original dataset loaded successfully!')
print('Dataset shape:', df.shape)
display(df.head())

## 3.1 Selecting Imputation Strategies

Age and Education are ordinal codes, Gender is binary categorical, and Nscore
is a continuous trait score. Median is used for Age, Education, and Nscore to
preserve their order/scale and stay robust to outliers. Mode is used for
Gender, since it's the only unordered categorical variable.


In [ ]:
df_clean = df.copy()

In [ ]:
for column in ["Age", "Gender", "Education", "Nscore"]:
      n_unique = df_clean[column].nunique()
      print(f"{column}: {n_unique} unique values")

## 3.2 Applying Imputation

Based on the strategy selected in the previous step, median imputation is
applied to `Age`, `Education`, and `Nscore`, while mode imputation is applied
to `Gender`. This replaces missing values with statistically appropriate
substitutes — the median for ordinal/continuous variables and the mode for
the binary categorical variable — without distorting the overall distribution
of each column.


In [ ]:
# Median imputation for numerical/ordinal variables
for column in ["Age", "Education", "Nscore"]:
    df_clean[column] = df_clean[column].fillna(
        df_clean[column].median()
    )

# Mode imputation for categorical variable
df_clean["Gender"] = df_clean["Gender"].fillna(
    df_clean["Gender"].mode()[0]
)

print("Missing values have been imputed.")

## 3.3 Verifying Imputation

After imputation, the dataset is checked again to confirm that the missing
values in Age, Gender, Education, and Nscore have been successfully handled.

In [ ]:
missing_after = df_clean.isnull().sum()
missing_after[missing_after > 0]

## 3.4 Preparing for Outlier Detection

Numerical variables are selected for outlier analysis because boxplots and
Z-scores are commonly used to identify unusually high or low numerical
observations.


In [ ]:
# Select continuous personality score variables
personality_cols = [
    "Nscore",
    "Escore",
    "Oscore",
    "Ascore",
    "Cscore",
    "Impulsive",
    "SS"
]

numeric_columns = personality_cols

print("Selected numerical variables:")
print(numeric_columns)

## 3.5 Outlier Detection Using Boxplots

Boxplots are used to visually identify potential outliers. Values located
beyond the whiskers of a boxplot may indicate observations that are unusually
high or low compared with the rest of the data.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

df_clean[numeric_columns].boxplot()

plt.xticks(rotation=90)
plt.title("Boxplot of Numerical Variables")
plt.ylabel("Values")
plt.show()

## 3.6 Outlier Detection Using Z-score

The Z-score method is used to identify numerical observations that are
unusually far from the mean. Observations with an absolute Z-score greater
than 3 are considered potential outliers for further investigation.

In [ ]:
from scipy.stats import zscore

z_scores = np.abs(zscore(df_clean[numeric_columns], nan_policy="omit"))

outlier_count = (z_scores > 3).sum(axis=0)

outlier_summary = pd.DataFrame({
    "Column": numeric_columns,
        "Outlier Count (|Z| > 3)": outlier_count
        })

print(outlier_summary)

## 3.7 Final Data Quality Check

The cleaned dataset is checked to confirm that missing values have been
handled and that the dataset retains its original number of observations and
variables. Potential outliers are reported separately rather than
automatically deleted.

In [ ]:
print("Final Shape:", df_clean.shape)

print("\nRemaining Missing Values:")
print(df_clean.isnull().sum().sum())

print("\nDuplicate Rows:")
print(df_clean.duplicated().sum())

## 3.8 Cleaning Sprint — Conclusion

The dataset was cleaned by identifying and handling missing values in Age,
Gender, Education, and Nscore using appropriate imputation strategies. Median
imputation was applied to Age, Education, and Nscore, while mode imputation
was applied to Gender, based on the ordinal/continuous versus categorical
nature of each variable. Boxplots and Z-score analysis were then used to
identify potential outliers across all numerical variables. Rather than
automatically removing these outliers, they were reported for further
investigation, since extreme values in this dataset may represent legitimate
observations rather than data errors. The final dataset retains its original
1885 observations and 32 variables, with no remaining missing values and no
duplicate rows.

## 3.9 Saving the Cleaned Dataset

Save the imputed dataset for use in the following project stages. Potential outliers are reported but are not automatically removed.

In [ ]:
output_path = Path('../Dataset/youth_drug_consumption_cleaned.csv')
df_clean.to_csv(output_path, index=False)
print('Cleaned dataset saved to:', output_path)